# CircleID — EVA02 FocusNet guided by ConvNeXt 0.937
### Baseline branch = frozen ConvNeXt Large (0.937 LB) — generates multi-warm labels
### FocusNet branch = EVA02 Large — trained with focus-picking loss
### Paper: FocusNet — Classifying Better by Focusing on Confusing Classes (Zhang et al. 2022)
### Key insight: ConvNeXt at 0.937 is a much stronger teacher than EVA02 Round 1
### Its confusion signal on Pen 3/7/8 is cleaner — better multi-warm labels

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import random
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = "/kaggle/input/datasets/mohamedsadokaissaoui/icdarrrrrrrrrrrrrrr/"

# Baseline branch — frozen ConvNeXt 0.937
CONVNEXT_MODEL_NAME    = 'timm/convnext_large.fb_in22k_ft_in1k'
CONVNEXT_BASELINE_PATH = '/kaggle/input/models/mohamedsadekissaoui/large5/pytorch/default/1/large5.pth'

# FocusNet branch — EVA02 Large
EVA02_MODEL_NAME = 'eva02_large_patch14_224'

print('Imports done.')

## Config

In [ ]:
class CFG:
    # Shared
    IMG_SIZE              = 224
    PAD_SIZE              = 120
    NUM_CLASSES           = 8
    DEVICE                = "cuda" if torch.cuda.is_available() else "cpu"
    SEED                  = 42
    MEAN                  = [0.485, 0.456, 0.406]
    STD                   = [0.229, 0.224, 0.225]

    # EVA02 FocusNet training
    BATCH_SIZE            = 32
    EPOCHS                = 25
    LR                    = 1e-5
    EARLY_STOPPING        = 8
    MIXUP_ALPHA           = 0.3

    # FocusNet loss weights — alpha=beta=1 per paper (Section 3.4)
    FOCUS_ALPHA           = 1.0
    FOCUS_BETA            = 1.0

    # Pseudo labeling
    PSEUDO_THRESHOLD      = 0.95   # raised — only very confident pseudo labels
    PSEUDO_EPOCHS         = 15
    PSEUDO_EARLY_STOPPING = 5
    MAX_PSEUDO_ROUNDS     = 5

print(f'Device: {CFG.DEVICE}')
if CFG.DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
def seed_everything(seed=42):
    import os
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

seed_everything(CFG.SEED)

## Data Loading & Cleaning

In [ ]:
train      = pd.read_csv(DATA_PATH + 'train.csv')
add_train  = pd.read_csv(DATA_PATH + 'additional_train.csv')
test       = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train: {len(train)} | Additional: {len(add_train)} | Test: {len(test)}')

In [ ]:
def fix_w50_labels(df):
    mask = (df['writer_id'] == 'W50') & (df['pen_id'] == 5)
    df.loc[mask, 'pen_id'] = 3
    return df

train     = fix_w50_labels(train)
add_train = fix_w50_labels(add_train)

train     = train[train['writer_id']     != 'W41'].reset_index(drop=True)
add_train = add_train[add_train['writer_id'] != 'W41'].reset_index(drop=True)

def has_ink(img_path, min_dark_pixels=50):
    try:
        img = np.array(Image.open(DATA_PATH + img_path).convert('L'))
        return int((img < 200).sum()) >= min_dark_pixels
    except:
        return False

train     = train[train['image_path'].apply(has_ink)].reset_index(drop=True)
add_train = add_train[add_train['image_path'].apply(has_ink)].reset_index(drop=True)

print(f'After cleaning — Train: {len(train)} | Additional: {len(add_train)}')

## Transforms
### ColorJitter removed — destroys ink texture signal critical for Pen 3/7/8
### RandomAdjustSharpness + RandomAutocontrast preserve stroke edge detail

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

MEAN     = CFG.MEAN
STD      = CFG.STD
PAD_SIZE = CFG.PAD_SIZE

# EVA02 FocusNet transforms — texture-preserving
train_transforms = transforms.Compose([
    transforms.Pad(padding=PAD_SIZE, fill=255),
    transforms.RandomCrop(CFG.IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    transforms.RandomAutocontrast(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transforms = transforms.Compose([
    transforms.Pad(padding=PAD_SIZE, fill=255),
    transforms.CenterCrop(CFG.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

eva_tta_list = [
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomRotation((90, 90)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomRotation((180, 180)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

# ConvNeXt TTA — same size/padding, used for final ensemble inference
convnext_tta_list = [
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomRotation((90, 90)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Pad(PAD_SIZE, fill=255), transforms.CenterCrop(CFG.IMG_SIZE), transforms.RandomRotation((180, 180)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

print('Transforms ready.')

## Dataset & DataLoaders

In [ ]:
class PenDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image    = Image.open(DATA_PATH + img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(self.df.loc[idx, 'pen_id']) - 1
        return image, torch.tensor(label, dtype=torch.long)


class TestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image    = Image.open(DATA_PATH + img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image


train_all = pd.concat([train, add_train]).reset_index(drop=True)
train_df, val_df = train_test_split(
    train_all, test_size=0.2, random_state=CFG.SEED, stratify=train_all['pen_id']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test)}')

In [ ]:
def make_loaders(train_df, val_df):
    train_loader = DataLoader(
        PenDataset(train_df, transform=train_transforms),
        batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        PenDataset(val_df, transform=val_transforms),
        batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders(train_df, val_df)
print('DataLoaders ready.')

## MixUp + CutMix

In [ ]:
def mixup_cutmix(x, y, alpha=0.3):
    lam        = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index      = torch.randperm(batch_size).to(x.device)

    if np.random.rand() < 0.5:
        mixed_x = lam * x + (1 - lam) * x[index]
    else:
        _, _, H, W = x.shape
        cx    = np.random.randint(W)
        cy    = np.random.randint(H)
        cut_w = int(W * np.sqrt(1 - lam))
        cut_h = int(H * np.sqrt(1 - lam))
        x1    = np.clip(cx - cut_w // 2, 0, W)
        x2    = np.clip(cx + cut_w // 2, 0, W)
        y1    = np.clip(cy - cut_h // 2, 0, H)
        y2    = np.clip(cy + cut_h // 2, 0, H)
        mixed_x = x.clone()
        mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))

    return mixed_x, y, y[index], lam

## FocusNet — Focus-Picking Loss
### Paper: Zhang et al. 2022 — Eq. 4-6 (multi-warm label), Eq. 12 (focus-picking loss)
###
### Multi-warm label (generated from frozen ConvNeXt logits each batch):
###   Classes with positive logit in ConvNeXt = confusing classes
###   GT class always included → normalize to sum=1
###   ConvNeXt at 0.937 gives cleaner confusion signal than a weaker baseline
###
### Focus-picking loss = L_cls + alpha*R_attention - beta*R_entropy
###   L_cls       : re-weighted CE — hard samples (EVA02 less confident than ConvNeXt) get larger loss
###   R_attention : forces EVA02 to assign higher probs to ConvNeXt-identified confusing classes
###   R_entropy   : penalizes overconfident distributions

In [ ]:
def make_multi_warm_label(baseline_logits, ground_truth, num_classes):
    """
    Eq. 4-6 from paper.
    baseline_logits : (B, C) — raw logits from frozen ConvNeXt
    ground_truth    : (B,)   — integer class indices
    Returns normalized multi-warm label (B, C)
    """
    # Eq. 4: l'_n = 1 if z^b_n > 0 else 0
    l_prime   = (baseline_logits > 0).float()                          # (B, C)

    # Eq. 5: add GT class, clip to 1
    gt_onehot = F.one_hot(ground_truth, num_classes).float()           # (B, C)
    l_double  = torch.clamp(l_prime + gt_onehot, max=1.0)             # (B, C)

    # Eq. 6: normalize rows to sum to 1
    row_sums  = l_double.sum(dim=1, keepdim=True).clamp(min=1.0)
    l         = l_double / row_sums                                    # (B, C)

    return l


class FocusPickingLoss(nn.Module):
    """
    Focus-picking loss — Eq. 12 from paper.
    L = L_cls + alpha * R_attention - beta * R_entropy
    """
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, focusnet_logits, baseline_logits, ground_truth):
        B, C = focusnet_logits.shape

        # Multi-warm label from frozen ConvNeXt — no grad needed
        with torch.no_grad():
            multi_warm = make_multi_warm_label(baseline_logits, ground_truth, C)  # (B, C)

        # Normal softmax of EVA02 FocusNet (d=0)
        p_focus_normal = F.softmax(focusnet_logits, dim=1)                        # (B, C)

        # d = softmax(EVA02) - softmax(ConvNeXt) — difficulty signal
        with torch.no_grad():
            p_baseline = F.softmax(baseline_logits, dim=1)                        # (B, C)
        d = p_focus_normal.detach() - p_baseline                                  # (B, C)

        # Re-weighted logits: z + d (Eq. 7)
        logits_reweighted = focusnet_logits + d                                   # (B, C)

        # L_cls: CE with re-weighted softmax (Eq. 9)
        # Negative d on GT = EVA02 less confident than ConvNeXt = hard sample = larger loss
        L_cls = F.cross_entropy(logits_reweighted, ground_truth)

        # R_attention: CE between EVA02 probs and multi-warm label (Eq. 10)
        # Forces EVA02 to assign higher probs to ConvNeXt-identified confusing classes
        R_attention = -(multi_warm * torch.log(p_focus_normal + 1e-8)).sum(dim=1).mean()

        # R_entropy: negative entropy (Eq. 11)
        # Penalizes overconfident distributions
        R_entropy = (p_focus_normal * torch.log(p_focus_normal + 1e-8)).sum(dim=1).mean()

        # Final loss (Eq. 12)
        loss = L_cls + self.alpha * R_attention - self.beta * R_entropy
        return loss


focus_criterion = FocusPickingLoss(alpha=CFG.FOCUS_ALPHA, beta=CFG.FOCUS_BETA)
print(f'FocusPickingLoss ready — alpha={CFG.FOCUS_ALPHA}, beta={CFG.FOCUS_BETA}')

## Evaluation Helper

In [ ]:
def evaluate(model, val_loader):
    """Plain CE evaluation — baseline not needed at inference time."""
    model.eval()
    ce = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(CFG.DEVICE)
            labels = labels.to(CFG.DEVICE)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = ce(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds       = torch.argmax(outputs, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

## FocusNet Training Loop
### ConvNeXt baseline is FROZEN throughout — only forward pass, never updated
### EVA02 FocusNet is trained from pretrained weights using focus-picking loss
### Both models share the same input batch — consistent d computation
### At inference: only EVA02 FocusNet needed, no extra compute

In [ ]:
def train_focusnet(focusnet_name, baseline_name, baseline_path,
                   train_loader, val_loader,
                   epochs, early_stopping, save_path,
                   focusnet_pretrained=True, resume_path=None):

    print(f'\n{"="*60}')
    print(f'Training EVA02 FocusNet — {epochs} epochs')
    print(f'Baseline (frozen): {baseline_name}')
    print(f'Baseline path:     {baseline_path}')
    print(f'{"="*60}')

    # --- Frozen ConvNeXt baseline ---
    baseline = timm.create_model(
        baseline_name, pretrained=False, num_classes=CFG.NUM_CLASSES
    ).to(CFG.DEVICE)
    # ConvNeXt benefits from channels_last memory format
    baseline = baseline.to(memory_format=torch.channels_last)
    ckpt     = torch.load(baseline_path, map_location=CFG.DEVICE)
    baseline.load_state_dict(ckpt['model'])
    baseline.eval()
    for p in baseline.parameters():
        p.requires_grad = False
    print(f'ConvNeXt baseline loaded and frozen ✅')

    # --- EVA02 FocusNet branch ---
    focusnet = timm.create_model(
        focusnet_name, pretrained=focusnet_pretrained, num_classes=CFG.NUM_CLASSES
    ).to(CFG.DEVICE)
    if resume_path is not None:
        ckpt2 = torch.load(resume_path, map_location=CFG.DEVICE)
        focusnet.load_state_dict(ckpt2['model'])
        print(f'FocusNet resumed from {resume_path}')

    optimizer = torch.optim.AdamW(focusnet.parameters(), lr=CFG.LR, weight_decay=0.05)

    # Warmup 3 epochs then cosine decay — important for ViT models
    warmup_epochs = 3
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.cuda.amp.GradScaler()

    best_acc = 0
    patience = 0

    for epoch in range(epochs):
        focusnet.train()
        baseline.eval()  # always keep baseline in eval
        train_loss, correct, total = 0, 0, 0

        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            images = images.to(CFG.DEVICE)
            labels = labels.to(CFG.DEVICE)

            # Apply same augmented batch to both branches
            images_aug, y_a, y_b, lam = mixup_cutmix(images, labels, CFG.MIXUP_ALPHA)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                # ConvNeXt: frozen forward pass — generates multi-warm labels
                with torch.no_grad():
                    baseline_logits = baseline(
                        images_aug.to(memory_format=torch.channels_last)
                    )

                # EVA02 FocusNet: forward pass
                focus_logits = focusnet(images_aug)

                # Focus-picking loss with MixUp blend
                loss = (
                    lam       * focus_criterion(focus_logits, baseline_logits, y_a) +
                    (1 - lam) * focus_criterion(focus_logits, baseline_logits, y_b)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(focusnet.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)
            preds       = torch.argmax(focus_logits, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

        train_loss /= total
        train_acc   = correct / total
        scheduler.step()
        val_loss, val_acc = evaluate(focusnet, val_loader)

        lr = optimizer.param_groups[0]['lr']
        print(
            f'Epoch {epoch+1:02d} | '
            f'Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | '
            f'Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f} | '
            f'LR {lr:.2e}'
        )

        if val_acc > best_acc:
            best_acc = val_acc
            patience = 0
            torch.save({'model': focusnet.state_dict()}, save_path)
            print(f'  ✅ New best: {best_acc:.4f}')
        else:
            patience += 1
            print(f'  No improvement ({patience}/{early_stopping})')
            if patience >= early_stopping:
                print('  Early stopping triggered.')
                break

    # Free baseline VRAM after training
    del baseline
    torch.cuda.empty_cache()

    print(f'Best Val Accuracy: {best_acc:.4f}')
    return best_acc

## TTA Inference

In [ ]:
def predict_with_tta(model_name, model_path, tta_list, use_channels_last=False):
    model = timm.create_model(model_name, pretrained=False, num_classes=CFG.NUM_CLASSES).to(CFG.DEVICE)
    if use_channels_last:
        model = model.to(memory_format=torch.channels_last)
    checkpoint = torch.load(model_path, map_location=CFG.DEVICE)
    model.load_state_dict(checkpoint['model'])
    model.eval()

    all_probs = None
    for i, tta_transform in enumerate(tta_list):
        tta_loader = DataLoader(
            TestDataset(test, transform=tta_transform),
            batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
        )
        preds = []
        with torch.no_grad():
            for images in tqdm(tta_loader, desc=f'TTA {i+1}/{len(tta_list)}'):
                images = images.to(CFG.DEVICE)
                if use_channels_last:
                    images = images.to(memory_format=torch.channels_last)
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                preds.append(torch.softmax(outputs, dim=1).cpu().numpy())
        preds     = np.concatenate(preds)
        all_probs = preds if all_probs is None else all_probs + preds

    del model
    torch.cuda.empty_cache()
    return all_probs / len(tta_list)

## Confusion Matrix Helper

In [ ]:
def plot_confusion_matrix(model_name, model_path, val_loader, title='', use_channels_last=False):
    model = timm.create_model(model_name, pretrained=False, num_classes=CFG.NUM_CLASSES).to(CFG.DEVICE)
    if use_channels_last:
        model = model.to(memory_format=torch.channels_last)
    checkpoint = torch.load(model_path, map_location=CFG.DEVICE)
    model.load_state_dict(checkpoint['model'])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Validation inference'):
            images = images.to(CFG.DEVICE)
            if use_channels_last:
                images = images.to(memory_format=torch.channels_last)
            with torch.cuda.amp.autocast():
                outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    pen_ids    = [f'Pen {i}' for i in range(1, CFG.NUM_CLASSES + 1)]

    cm      = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(title, fontsize=15, fontweight='bold')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=pen_ids, yticklabels=pen_ids, ax=axes[0])
    axes[0].set_title('Raw Counts'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=pen_ids, yticklabels=pen_ids, ax=axes[1])
    axes[1].set_title('Normalized'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'cm_{title.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n--- Per-class accuracy ---')
    for i in range(CFG.NUM_CLASSES):
        mask = all_labels == i
        acc  = (all_preds[mask] == i).mean()
        print(f'  Pen {i+1}: {acc:.3f}  {"█" * int(acc * 20)}  ({mask.sum()} samples)')

    print('\n--- Top confused pairs ---')
    cm_off = cm.copy()
    np.fill_diagonal(cm_off, 0)
    for idx in np.argsort(cm_off.flatten())[::-1][:5]:
        t, p = idx // CFG.NUM_CLASSES, idx % CFG.NUM_CLASSES
        if cm_off[t, p] > 0:
            print(f'  True Pen {t+1} → Predicted Pen {p+1}: {cm_off[t,p]} times')

    print('\n--- Classification Report ---')
    print(classification_report(all_labels, all_preds, target_names=pen_ids))

    del model
    torch.cuda.empty_cache()
    return all_preds, all_labels

## Train EVA02 FocusNet — ConvNeXt 0.937 as Frozen Baseline

In [ ]:
save_path_focusnet = 'eva02_focusnet.pth'

acc_focusnet = train_focusnet(
    focusnet_name      = EVA02_MODEL_NAME,
    baseline_name      = CONVNEXT_MODEL_NAME,
    baseline_path      = CONVNEXT_BASELINE_PATH,
    train_loader       = train_loader,
    val_loader         = val_loader,
    epochs             = CFG.EPOCHS,
    early_stopping     = CFG.EARLY_STOPPING,
    save_path          = save_path_focusnet,
    focusnet_pretrained = True,
)
print(f'\nFocusNet best val acc: {acc_focusnet:.4f}')

In [ ]:
plot_confusion_matrix(EVA02_MODEL_NAME, save_path_focusnet, val_loader, title='EVA02 FocusNet Round 1')

## Pseudo Labeling Rounds 2-5
### Baseline stays as frozen ConvNeXt throughout all rounds — its confusion signal doesn't change
### Only the FocusNet branch improves each round
### Pen 3/7/8 excluded — even ConvNeXt is unreliable on these at pseudo label time
### Threshold 0.95 — only very confident predictions

In [ ]:
current_model_path = save_path_focusnet
best_acc           = acc_focusnet

for round_num in range(2, CFG.MAX_PSEUDO_ROUNDS + 1):
    print(f'\n{"="*50}')
    print(f'FOCUSNET PSEUDO LABELING ROUND {round_num}')
    print(f'{"="*50}')

    # Use current best FocusNet for pseudo label generation
    test_probs = predict_with_tta(EVA02_MODEL_NAME, current_model_path, eva_tta_list)
    max_probs  = test_probs.max(axis=1)
    pseudo_ids = test_probs.argmax(axis=1) + 1

    # Exclude Pen 3/7/8 — confusing classes, pseudo labels would add noise
    mask_not_378   = ~np.isin(pseudo_ids, [3, 7, 8])
    mask_confident = max_probs >= CFG.PSEUDO_THRESHOLD
    mask           = mask_confident & mask_not_378

    print(f'Confident samples (excl 3/7/8): {mask.sum()} / {len(test)} ({mask.mean()*100:.1f}%)')
    print(f'Confidence — mean: {max_probs.mean():.4f} | median: {np.median(max_probs):.4f} | max: {max_probs.max():.4f}')
    print(f'Pseudo label distribution:')
    print(pd.Series(pseudo_ids[mask]).value_counts().sort_index())

    if mask.sum() == 0:
        print('No confident samples — stopping early.')
        break

    pseudo_df              = test[mask].copy().reset_index(drop=True)
    pseudo_df['pen_id']    = pseudo_ids[mask]
    pseudo_df['writer_id'] = -1

    train_round = pd.concat([train_df, pseudo_df]).reset_index(drop=True)
    train_loader_r, _ = make_loaders(train_round, val_df)
    print(f'Training on {len(train_round)} samples ({len(pseudo_df)} pseudo)')

    save_path_round = f'eva02_focusnet_round{round_num}.pth'

    # ConvNeXt baseline stays frozen — FocusNet resumes from current best
    acc_round = train_focusnet(
        focusnet_name       = EVA02_MODEL_NAME,
        baseline_name       = CONVNEXT_MODEL_NAME,
        baseline_path       = CONVNEXT_BASELINE_PATH,  # always the same ConvNeXt
        train_loader        = train_loader_r,
        val_loader          = val_loader,
        epochs              = CFG.PSEUDO_EPOCHS,
        early_stopping      = CFG.PSEUDO_EARLY_STOPPING,
        save_path           = save_path_round,
        focusnet_pretrained = False,
        resume_path         = current_model_path,
    )

    print(f'Round {round_num} val acc: {acc_round:.4f} (prev best: {best_acc:.4f})')
    plot_confusion_matrix(EVA02_MODEL_NAME, save_path_round, val_loader, title=f'FocusNet Round {round_num}')

    if acc_round >= best_acc:
        best_acc           = acc_round
        current_model_path = save_path_round
        print(f'✅ New best: {save_path_round}')
    else:
        print('⚠️  No improvement — keeping previous model, continuing.')

print(f'\nFocusNet training complete.')
print(f'Best model: {current_model_path}')
print(f'Best val acc: {best_acc:.4f}')

## EVA02 FocusNet Standalone Submission

In [ ]:
print('Running FocusNet final TTA inference...')
eva_probs  = predict_with_tta(EVA02_MODEL_NAME, current_model_path, eva_tta_list)
eva_labels = np.argmax(eva_probs, axis=1) + 1

pd.DataFrame({
    'image_id': test['image_id'],
    'pen_id':   eva_labels
}).to_csv('submission_eva02_focusnet.csv', index=False)

print('Saved submission_eva02_focusnet.csv')
print('EVA02 FocusNet distribution:')
print((pd.Series(eva_labels).value_counts() / len(eva_labels) * 100).sort_index().round(2))

## Ensemble — EVA02 FocusNet + ConvNeXt 0.937

In [ ]:
print('Running ConvNeXt TTA inference...')
convnext_probs  = predict_with_tta(
    CONVNEXT_MODEL_NAME, CONVNEXT_BASELINE_PATH,
    convnext_tta_list, use_channels_last=True
)
convnext_labels = np.argmax(convnext_probs, axis=1) + 1
print('ConvNeXt distribution:')
print((pd.Series(convnext_labels).value_counts() / len(convnext_labels) * 100).sort_index().round(2))

In [ ]:
# Equal weight ensemble
ensemble_probs  = (eva_probs + convnext_probs) / 2
ensemble_labels = np.argmax(ensemble_probs, axis=1) + 1

pd.DataFrame({
    'image_id': test['image_id'],
    'pen_id':   ensemble_labels
}).to_csv('submission_ensemble_equal.csv', index=False)

print('Equal weight ensemble distribution:')
print((pd.Series(ensemble_labels).value_counts() / len(ensemble_labels) * 100).sort_index().round(2))
print('Saved submission_ensemble_equal.csv')

In [ ]:
# Weighted ensembles
for eva_w, cnxt_w in [(0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4)]:
    probs  = eva_w * eva_probs + cnxt_w * convnext_probs
    labels = np.argmax(probs, axis=1) + 1
    dist   = (pd.Series(labels).value_counts() / len(labels) * 100).sort_index().round(2)
    fname  = f'submission_eva{int(eva_w*10)}_cnxt{int(cnxt_w*10)}.csv'
    pd.DataFrame({'image_id': test['image_id'], 'pen_id': labels}).to_csv(fname, index=False)
    pen3 = dist.get(3, 0)
    pen7 = dist.get(7, 0)
    print(f'EVA={eva_w} CNXT={cnxt_w} | Pen3: {pen3:.1f}% Pen7: {pen7:.1f}% | Saved {fname}')

In [ ]:
print('\n=== Summary ===')
print(f'EVA02 FocusNet best val acc: {best_acc:.4f}')
print(f'ConvNeXt baseline LB score:  0.937')
print(f'\nFiles saved:')
print('  submission_eva02_focusnet.csv  — FocusNet standalone')
print('  submission_ensemble_equal.csv  — 50/50 ensemble')
print('  submission_eva3_cnxt7.csv      — 30/70 ConvNeXt heavy')
print('  submission_eva4_cnxt6.csv      — 40/60 ConvNeXt heavy')
print('  submission_eva5_cnxt5.csv      — 50/50')
print('  submission_eva6_cnxt4.csv      — 60/40 FocusNet heavy')
print('\nRecommended submit order:')
print('  1. submission_eva02_focusnet.csv — check FocusNet standalone LB')
print('  2. submission_ensemble_equal.csv — check ensemble')
print('  3. Weighted versions based on FocusNet standalone score')